<a href="https://colab.research.google.com/github/VijayaRagav07/MMB/blob/main/OptimizedMMB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import gc
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("🚀 GPU:", tf.config.list_physical_devices('GPU'))

BASE_PATH = "/content/drive/MyDrive/MMB datsets"

MODALITIES = {
    "ultrasound": "Ultrasound Images_MSI",
    "histopath": "Histopathological_MSI",
    "xray": "Chest_XRay_MSI"
}

Mounted at /content/drive
🚀 GPU: []


In [ ]:
IMG_SIZE = 224

def load_images(path):
    images = []
    labels = []

    class_names = ["Benign", "Malignant"]

    print(f"📂 Loading from: {path}")

    for label, class_name in enumerate(class_names):
        class_path = os.path.join(path, class_name)

        if not os.path.exists(class_path):
            print("❌ Missing:", class_path)
            continue

        print(f"   → {class_name}...")

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            img = cv2.imread(img_path)
            if img is None:
                continue

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

            rgb = img / 255.0
            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV) / 255.0
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            jet = cv2.applyColorMap(gray, cv2.COLORMAP_JET) / 255.0

            combined = np.concatenate([rgb, hsv, jet], axis=-1)

            images.append(combined)
            labels.append(label)

    images = np.array(images, dtype=np.float32)
    labels = np.array(labels)

    print("✅ Loaded:", images.shape)

    return images, labels

In [ ]:
def build_model():
    inputs = layers.Input(shape=(224,224,9))

    x = layers.Conv2D(32, 3, activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)   # 🔥 better than Flatten

    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
def train_on_modality(modality):
    print(f"\n🚀 TRAINING ON {modality.upper()}")

    path = os.path.join(BASE_PATH, MODALITIES[modality])
    X, y = load_images(path)

    idx = np.random.permutation(len(X))
    X, y = X[idx], y[idx]

    split = int(0.8 * len(X))
    X_train, X_val = X[:split], X[split:]
    y_train, y_val = y[:split], y[split:]

    model = build_model()

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=16,
        verbose=1
    )

    # free train data
    del X, y, X_train, X_val, y_train, y_val
    gc.collect()

    return model

In [ ]:
def evaluate_model(model, modality):
    print(f"\n🔍 Testing on {modality.upper()}")

    path = os.path.join(BASE_PATH, MODALITIES[modality])
    X_test, y_test = load_images(path)

    preds = model.predict(X_test)
    preds_binary = (preds > 0.5).astype(int)

    print("\n📊 Classification Report:")
    print(classification_report(y_test, preds_binary))

    print("📊 Confusion Matrix:")
    print(confusion_matrix(y_test, preds_binary))

    try:
        auc = roc_auc_score(y_test, preds)
        print("📈 ROC-AUC:", auc)
    except:
        print("ROC-AUC not available")

    del X_test, y_test
    gc.collect()

In [ ]:
for train_mod in MODALITIES:
    print("\n" + "="*50)
    print(f"🔥 TRAIN ON {train_mod.upper()}")
    print("="*50)

    model = train_on_modality(train_mod)

    for test_mod in MODALITIES:
        evaluate_model(model, test_mod)

    tf.keras.backend.clear_session()
    gc.collect()


🔥 TRAIN ON ULTRASOUND

🚀 TRAINING ON ULTRASOUND
📂 Loading from: /content/drive/MyDrive/MMB datsets/Ultrasound Images_MSI
   → Benign...
   → Malignant...
✅ Loaded: (806, 224, 224, 9)
Epoch 1/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 56s 1s/step - accuracy: 0.4938 - loss: 0.6976 - val_accuracy: 0.4753 - val_loss: 0.6936
Epoch 2/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.5233 - loss: 0.6919 - val_accuracy: 0.4877 - val_loss: 0.6903
Epoch 3/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.5373 - loss: 0.6850 - val_accuracy: 0.4753 - val_loss: 0.6917
Epoch 4/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - accuracy: 0.5761 - loss: 0.6820 - val_accuracy: 0.6420 - val_loss: 0.6795
Epoch 5/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.5823 - loss: 0.6767 - val_accuracy: 0.5802 - val_loss: 0.6748
Epoch 6/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.6087 - loss: 0.6671 - val_accuracy: 0.6605 - val_loss: 0.6619
Epoch 7/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy

In [ ]:
!pip install colab-pdf

ERROR: Could not find a version that satisfies the requirement colab-pdf (from versions: none)
ERROR: No matching distribution found for colab-pdf


In [ ]:
from colab_pdf import colab_pdf
colab_pdf('OptimizedMMB.ipynb')

ModuleNotFoundError: No module named 'colab_pdf'